In [1]:
from client import AGRClient
from downloader import Downloader
from normalizer import GeneIndex, load_gene_index

from pathlib import Path
import pandas as pd

In [2]:
c = AGRClient()
dl = Downloader()

In [3]:
files = await c.list_downloads()

In [4]:
orthology_tsv = next((f for f in files if f.dataType == "ORTHOLOGY-ALLIANCE" and f.fileType == "TSV"), None)
gene_tsv = next((f for f in files if f.dataType == "GENE" and f.dataSubType == 'COMBINED' and f.fileType == "TSV"), None)

if orthology_tsv:
    await dl.download(orthology_tsv.s3Url, Path("/lab01/Projects/Lionel_Projects/alliance_wrapper/cache/orthology-alliance.tsv.gz"))

if gene_tsv:
    await dl.download(gene_tsv.s3Url, Path("/lab01/Projects/Lionel_Projects/alliance_wrapper/cache/gene.tsv.gz"))

In [5]:
pd.read_csv("/lab01/Projects/Lionel_Projects/alliance_wrapper/cache/orthology-alliance.tsv.gz", sep="\t", compression="gzip", comment="#")

,Gene1ID,Gene1Symbol,Gene1SpeciesTaxonID,Gene1SpeciesName,Gene2ID,Gene2Symbol,Gene2SpeciesTaxonID,Gene2SpeciesName,Algorithms,AlgorithmsMatch,OutOfAlgorithms,IsBestScore,IsBestRevScore
0,Xenbase:XB-GENE-29081103,or5ar54,NCBITaxon:8364,NaN,MGI:3030366,Or13a21,NCBITaxon:10090,NaN,OMA|PANTHER|SonicParanoid,3,9,Yes,Yes
1,MGI:2177485,Or12d2,NCBITaxon:10090,NaN,Xenbase:XB-GENE-29083495,or5ar4b,NCBITaxon:8364,NaN,OMA|OrthoFinder|OrthoInspector|PhylomeDB,4,9,No,Yes
2,MGI:2177485,Or12d2,NCBITaxon:10090,NaN,Xenbase:XB-GENE-29083583,or5ar4,NCBITaxon:8364,NaN,InParanoid|OrthoFinder|OrthoInspector|PhylomeD...,5,9,Yes,Yes
3,HGNC:2511,CTNNA3,NCBITaxon:9606,NaN,Xenbase:XB-GENE-29096671,ctnna3,NCBITaxon:8364,NaN,Ensembl Compara|PhylomeDB,2,9,Yes,Yes
4,ZFIN:ZDB-GENE-131127-50,si:dkey-261m9.7,NCBITaxon:7955,NaN,Xenbase:XB-GENE-29077871,LOC100485548,NCBITaxon:8364,NaN,Hieranoid|OMA|OrthoFinder|OrthoInspector|Sonic...,5,9,Yes,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...
987009,WB:WBGene00009999,srsx-35,NCBITaxon:6239,NaN,RGD:7579074,LOC102555599,NCBITaxon:10116,NaN,Hieranoid|SonicParanoid,2,9,Yes,Yes
987010,HGNC:28527,NXPE1,NCBITaxon:9606,NaN,Xenbase:XB-GENE-29087443,LOC101734002,NCBITaxon:8364,NaN,Ensembl Compara|OrthoFinder|PANTHER|PhylomeDB,4,9,No,Yes
987011,HGNC:26331,NXPE2,NCBITaxon:9606,NaN,Xenbase:XB-GENE-29082663,LOC100495331,NCBITaxon:8364,NaN,Ensembl Compara|Hieranoid|InParanoid|OMA|Ortho...,9,9,Yes,Yes
987012,WB:WBGene00018880,acc-3,NCBITaxon:6239,NaN,FB:FBgn0036727,SecCl,NCBITaxon:7227,NaN,InParanoid|PANTHER|PhylomeDB,3,9,Yes,No


In [6]:
pd.read_csv("/lab01/Projects/Lionel_Projects/alliance_wrapper/cache/gene.tsv.gz", 
            sep="\t", 
            compression="gzip", 
            comment="#", 
            usecols=['Taxon', 'SpeciesName', 'GeneId', 'GeneSymbol', 'GeneSynonyms'],
)

,Taxon,SpeciesName,GeneId,GeneSymbol,GeneSynonyms
0,NCBITaxon:10090,Mus musculus,MGI:6008918,Tssr81151,mm_75357.1
1,NCBITaxon:10090,Mus musculus,MGI:6008936,Tssr81169,mm_75375.1
2,NCBITaxon:10090,Mus musculus,MGI:6008972,Tssr81205,mm_75411.1
3,NCBITaxon:10090,Mus musculus,MGI:6008978,Tssr81211,mm_75417.1
4,NCBITaxon:10090,Mus musculus,MGI:6008985,Tssr81218,mm_75424.1
...,...,...,...,...,...
914078,NCBITaxon:7227,Drosophila melanogaster,FB:FBgn0290431,Vps2,complementation group 8|Dvps2|CG14542|vps2|vac...
914079,NCBITaxon:7955,Danio rerio,ZFIN:ZDB-GENE-260127-1,pcdh2g1.1,NaN
914080,NCBITaxon:7955,Danio rerio,ZFIN:ZDB-GENE-260310-14,nilt17,NaN
914081,NCBITaxon:7955,Danio rerio,ZFIN:ZDB-GENE-260310-22,nilt27,NaN


In [7]:
gi = load_gene_index(Path("/lab01/Projects/Lionel_Projects/alliance_wrapper/cache/gene.tsv.gz"))

In [ ]:
gene_queries = [
    "TP53",
    "BRCA1",
    "EGFR",
    "RPA1",
    "CELSR3",
    "miss!"
]

df = gi.normalize(gene_queries, limit = 10, taxon = "human", case_insensitive = True)

,query,match_kind,Taxon,SpeciesName,GeneId,GeneSymbol,GeneSynonyms,GeneSystematicName,GeneSecondaryIds,GeneCrossReferences,GeneBioTypeId,GeneBioTypeName,GeneAllianceAutomatedDescription,GeneMODAutomatedDescription,GeneMODDescription,Assembly,Chromosome,StartPosition,EndPosition,Strand
0,TP53,OFFICIAL_SYMBOL,NCBITaxon:9606,Homo sapiens,HGNC:11998,TP53,LFS1|TRP53|FLJ92943|P53|phosphoprotein p53|cel...,NaN,RGD:70502,RGD:70502|ENSEMBL:ENSG00000141510|NCBI_Gene:71...,SO:0001217,protein_coding_gene,"Enables several functions, including DNA bindi...",NaN,This gene encodes a tumor suppressor protein c...,GRCh38,17,7661779,7687550,-
1,TP53,SYNONYM,NCBITaxon:9606,Homo sapiens,HGNC:11999,TP53BP1,p202|53BP1|FLJ41424|MGC138366|tumor suppressor...,NaN,RGD:1317577,UniProtKB:B7Z3E7|UniProtKB:Q7Z3U4|UniProtKB:M0...,SO:0001217,protein_coding_gene,"Enables several functions, including histone r...",NaN,This gene encodes a protein that functions in ...,GRCh38,15,43403061,43510728,-
2,BRCA1,OFFICIAL_SYMBOL,NCBITaxon:9606,Homo sapiens,HGNC:1100,BRCA1,BROVCA1|IRIS|PSCP|BRCAI|BRCC1|RNF53|breast can...,NaN,RGD:69132,RGD:69132|ENSEMBL:ENSG00000012048|NCBI_Gene:67...,SO:0001217,protein_coding_gene,"Enables several functions, including enzyme bi...",NaN,This gene encodes a 190 kD nuclear phosphoprot...,GRCh38,17,43044292,43170327,-
3,EGFR,OFFICIAL_SYMBOL,NCBITaxon:9606,Homo sapiens,HGNC:3236,EGFR,ERBB|HER1|mENA|ERBB1|PIG61|receptor tyrosine-p...,NaN,RGD:69152,RGD:69152|ENSEMBL:ENSG00000146648|NCBI_Gene:19...,SO:0001217,protein_coding_gene,"Enables several functions, including actin fil...",NaN,The protein encoded by this gene is a transmem...,GRCh38,7,55018820,55211628,+
4,RPA1,OFFICIAL_SYMBOL,NCBITaxon:9606,Homo sapiens,HGNC:10289,RPA1,HSSB|RF-A|RP-A|REPA1|RPA70|MST075|replication ...,NaN,RGD:1316541,RGD:1316541|ENSEMBL:ENSG00000132383|NCBI_Gene:...,SO:0001217,protein_coding_gene,Enables G-rich strand telomeric DNA binding ac...,NaN,This gene encodes the largest subunit of the h...,GRCh38,17,1829702,1900082,+
5,RPA1,SYNONYM,NCBITaxon:9606,Homo sapiens,HGNC:17264,POLR1A,A190|RPA1|RPO14|RPA194|RPO1-4|FLJ21915|MGC8796...,NaN,RGD:1345694,RGD:1345694|ENSEMBL:ENSG00000068654|NCBI_Gene:...,SO:0001217,protein_coding_gene,"Enables several functions, including DNA-direc...",NaN,The protein encoded by this gene is the larges...,GRCh38,2,86020216,86106155,-
6,CELSR3,OFFICIAL_SYMBOL,NCBITaxon:9606,Homo sapiens,HGNC:3230,CELSR3,"cadherin, EGF LAG seven-pass G-type receptor 3...",NaN,RGD:1342685,UniProtKB:L0R6E8|UniProtKB:O75092|OMIM:604264|...,SO:0001217,protein_coding_gene,Predicted to enable G protein-coupled receptor...,NaN,"This gene belongs to the flamingo subfamily, w...",GRCh38,3,48636463,48662886,-
7,miss!,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
await c.aclose()
await dl.aclose()